In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np

In [3]:
county_geo = (
    gpd.read_file('../data/shapefiles/US_COUNTY_2022.gpkg')
    .to_crs('EPSG:4326')
)

In [4]:
plant_locations_by_year = {}

for load_year in range(2016, 2024):
    plant_geo = pd.read_excel(f"../data/power_plant_locations/2___Plant_Y{load_year}.xlsx")
    plant_geo.columns = plant_geo.loc[0]
    plant_geo = (
        plant_geo.loc[1:]
        .copy()
        .reset_index(drop=True)
        [['Utility ID', 'Utility Name', 'Plant Code', 'Plant Name', 'County', 'State', 'Latitude', 'Longitude']]
    )
    plant_geo.loc[plant_geo.Longitude == ' ', 'Longitude'] = np.nan
    plant_geo.loc[plant_geo.Latitude == ' ', 'Latitude'] = np.nan
    plant_geo['geometry'] = gpd.points_from_xy(plant_geo['Longitude'], plant_geo['Latitude'])
    plant_geo = gpd.GeoDataFrame(plant_geo, geometry='geometry', crs='EPSG:4326')
    
    plant_county_geo = gpd.sjoin(
        plant_geo,
        county_geo[['rb', 'geometry', 'STCODE']],
        how='inner',
        predicate='intersects'
    )

    plant_locations_by_year[load_year] = (
        plant_county_geo.set_index(['Plant Code', 'Plant Name'])
        [['rb', 'geometry', 'Longitude', 'Latitude', 'State', 'STCODE', 'County']]
    )

In [5]:
plant_locations = pd.concat(
    {k:v['rb'] for k,v in plant_locations_by_year.items()},
    axis=1
)
plant_locations['fips_all_years'] = (
    plant_locations.apply(lambda x: [fips for fips in list(set(x.tolist())) if not pd.isna(fips)], axis=1)
)
unmatched_plants = plant_locations.loc[plant_locations.fips_all_years.apply(len) > 1].copy()

In [7]:
plant_states = pd.concat(
    {k:v['State'] for k,v in plant_locations_by_year.items()},
    axis=1
)
plant_states['states_all_years'] = (
    plant_states.apply(lambda x: [fips for fips in list(set(x.tolist())) if not pd.isna(fips)], axis=1)
)

In [79]:
plant_counties = pd.concat(
    {k:v['County'] for k,v in plant_locations_by_year.items()},
    axis=1
)
plant_counties['counties_all_years'] = (
    plant_counties.apply(lambda x: [fips for fips in list(set(x.tolist())) if not pd.isna(fips)], axis=1)
)

In [92]:
unmatched = (
    unmatched_plants.merge(plant_states[['states_all_years']], left_index=True, right_index=True, how='left')
    .merge(plant_counties[['counties_all_years']], left_index=True, right_index=True, how='left')
    [['fips_all_years', 'states_all_years', 'counties_all_years']]
    .explode('fips_all_years')
    .rename(columns={'fips_all_years': 'fips'})
    .reset_index()
    .merge(county_geo[['rb', 'NAME']], left_on='fips', right_on='rb', how='left')
)

In [114]:
new_matched = (
    unmatched.loc[(
        unmatched.apply(axis=1, func=lambda x: (x['NAME'].replace('-', ' ') in x['counties_all_years']))
    )]
    .drop_duplicates(subset=['Plant Code', 'Plant Name'], keep=False)
    .set_index(['Plant Code', 'Plant Name'])
    [['fips']]
)

In [138]:
plant_locations_all = (
    pd.concat([
        (
            plant_locations.loc[plant_locations.fips_all_years.apply(len) == 1]
            .assign(fips=lambda x: x['fips_all_years'].str[0])
            [['fips']]
        ),
        new_matched
    ])
    .rename(columns={'fips': 'FIPS'})
)
plant_locations_all = pd.concat([
    plant_locations_all,
    (
        plant_locations.loc[~plant_locations.index.isin(plant_locations_all.index)]
        .rename(columns={2023: 'FIPS'})
        [['FIPS']]
    )
])

In [140]:
plant_locations_all.to_csv('../data/power_plant_locations/power_plant_counties.csv')